**Análisis de ventas mediante visualizaciones en Databricks                                                                                                                                                                             
**Brayan Andres Martin Diaz Principe**

**Paso 1. Empezamos cargando y revisando las columnas del archivo**

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, monotonically_increasing_id

spark = SparkSession.builder.appName("Villarica_Diaz").getOrCreate()

# Cargamos tu archivo CSV desde tu volumen
df = spark.read.csv(
    "/Volumes/workspace/villarica/disco_a/RESULTADOS-Villarica.csv",
    header=True,
    sep=",",
    inferSchema=True
)

# Ampliamos el dataset a 3,500 registros iniciales con ID único
df = (df.union(df).limit(3500).withColumn("id_fact_venta", monotonically_increasing_id() + 1)
)

display(df.limit(5))

id_fact_venta,id_venta,fecha,producto,categoria,cantidad,precio,metodo_pago
1,1,2026-10-01,Torta de chocolate,Pasteles y tortas,1,7.0,Efectivo
2,1,2026-10-01,Keke de naranja,Pasteles y tortas,1,4.0,Efectivo
3,2,2026-10-01,Té verde,Tés e infusiones,4,3.5,Efectivo
4,3,2026-10-01,Café americano,Cafés,6,4.0,Efectivo
5,4,2026-10-01,Té verde,Tés e infusiones,2,3.5,Yape


**Paso 2. Acá revisamos la estructura de los datos**

In [0]:
# Mostrar tipos de datos
df.printSchema()

# Mostrar estadísticas generales
df.describe().show()

# Contar registros
print("Total de registros:", df.count())

root
 |-- id_fact_venta: long (nullable = false)
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)
 |-- metodo_pago: string (nullable = true)

+-------+------------------+------------------+--------------------+----------------+-----------------+------------------+-----------+
|summary|     id_fact_venta|          id_venta|            producto|       categoria|         cantidad|            precio|metodo_pago|
+-------+------------------+------------------+--------------------+----------------+-----------------+------------------+-----------+
|  count|              3500|              3500|                3500|            3500|             3500|              3500|       3500|
|   mean|            1750.5| 537.8831428571428|                NULL|            NULL|             2.32| 4.581999999999996|       NU

**Paso 3. Procedemos a eliminar espacios innecesarios para que se vea más limpio**

In [0]:
# Limpiar espacios en columnas de texto
df = (df
    .withColumn("producto", trim(col("producto")))
    .withColumn("categoria", trim(col("categoria")))
    .withColumn("metodo_pago", trim(col("metodo_pago")))
)

df.show(5, truncate=False)

+-------------+--------+----------+------------------+-----------------+--------+------+-----------+
|id_fact_venta|id_venta|fecha     |producto          |categoria        |cantidad|precio|metodo_pago|
+-------------+--------+----------+------------------+-----------------+--------+------+-----------+
|1            |1       |2026-10-01|Torta de chocolate|Pasteles y tortas|1       |7.0   |Efectivo   |
|2            |1       |2026-10-01|Keke de naranja   |Pasteles y tortas|1       |4.0   |Efectivo   |
|3            |2       |2026-10-01|Té verde          |Tés e infusiones |4       |3.5   |Efectivo   |
|4            |3       |2026-10-01|Café americano    |Cafés            |6       |4.0   |Efectivo   |
|5            |4       |2026-10-01|Té verde          |Tés e infusiones |2       |3.5   |Yape       |
+-------------+--------+----------+------------------+-----------------+--------+------+-----------+
only showing top 5 rows


**Paso 4. Identificamos y tratamos valores nulos**

In [0]:
# Revisar valores nulos por columna
df.select([col(c).isNull().cast("int").alias(c)
    for c in df.columns
]).groupBy().sum().show()

df = df.na.drop(
    subset=[
        "id_fact_venta",
        "id_venta",
        "fecha",
        "producto",
        "categoria",
        "cantidad",
        "precio",
        "metodo_pago"
    ]
)

+------------------+-------------+----------+-------------+--------------+-------------+-----------+----------------+
|sum(id_fact_venta)|sum(id_venta)|sum(fecha)|sum(producto)|sum(categoria)|sum(cantidad)|sum(precio)|sum(metodo_pago)|
+------------------+-------------+----------+-------------+--------------+-------------+-----------+----------------+
|                 0|            0|         0|            0|             0|            0|          0|               0|
+------------------+-------------+----------+-------------+--------------+-------------+-----------+----------------+



**Paso 5. Corregimos tipos de datos**

In [0]:
df = (df
    .withColumn("id_fact_venta", col("id_fact_venta").cast("integer"))
    .withColumn("id_venta", col("id_venta").cast("integer"))
    .withColumn("fecha", to_date(col("fecha"), "yyyy-MM-dd"))
    .withColumn("cantidad", col("cantidad").cast("integer"))
    .withColumn("precio", col("precio").cast("decimal(12,2)"))
)

df.printSchema()

root
 |-- id_fact_venta: integer (nullable = false)
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: decimal(12,2) (nullable = true)
 |-- metodo_pago: string (nullable = true)



**Paso 6. Filtramos registros incorrectos**

In [0]:
df = df.filter(
    (col("id_fact_venta") > 0) &
    (col("id_venta") > 0) &
    (col("fecha").isNotNull()) &
    (col("producto") != "") &
    (col("categoria") != "") &
    (col("cantidad") > 0) &
    (col("precio") > 0) &
    (col("metodo_pago") != "")
)

df.show(10, truncate=False)

+-------------+--------+----------+--------------------------------+-----------------+--------+------+-----------+
|id_fact_venta|id_venta|fecha     |producto                        |categoria        |cantidad|precio|metodo_pago|
+-------------+--------+----------+--------------------------------+-----------------+--------+------+-----------+
|1            |1       |2026-10-01|Torta de chocolate              |Pasteles y tortas|1       |7.00  |Efectivo   |
|2            |1       |2026-10-01|Keke de naranja                 |Pasteles y tortas|1       |4.00  |Efectivo   |
|3            |2       |2026-10-01|Té verde                        |Tés e infusiones |4       |3.50  |Efectivo   |
|4            |3       |2026-10-01|Café americano                  |Cafés            |6       |4.00  |Efectivo   |
|5            |4       |2026-10-01|Té verde                        |Tés e infusiones |2       |3.50  |Yape       |
|6            |4       |2026-10-01|Galleta con chispas de chocolate|Postres y du

**Paso 7. Eliminamos registros duplicados para que así se vea más limpio**

In [0]:
df = df.dropDuplicates()

print("Registros después de eliminar duplicados:", df.count())

Registros después de eliminar duplicados: 3500


**Paso 8. Crear una nueva columna calculada**

In [0]:
from pyspark.sql.functions import sum as spark_sum

df = df.withColumn(
    "importe_total",
    col("cantidad") * col("precio")
)

df.select(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).show()

+--------------+------------+
|Total_Unidades|Total_Ventas|
+--------------+------------+
|          8120|    33265.80|
+--------------+------------+



**Paso 9. Seleccionar columnas necesarias**

In [0]:
df = df.select(
    "id_fact_venta",
    "id_venta",
    "fecha",
    "producto",
    "categoria",
    "cantidad",
    "precio",
    "importe_total",
    "metodo_pago"
)

df.show(10, truncate=False)

+-------------+--------+----------+--------------------------------+-----------------+--------+------+-------------+-----------+
|id_fact_venta|id_venta|fecha     |producto                        |categoria        |cantidad|precio|importe_total|metodo_pago|
+-------------+--------+----------+--------------------------------+-----------------+--------+------+-------------+-----------+
|1            |1       |2026-10-01|Torta de chocolate              |Pasteles y tortas|1       |7.00  |7.00         |Efectivo   |
|2            |1       |2026-10-01|Keke de naranja                 |Pasteles y tortas|1       |4.00  |4.00         |Efectivo   |
|3            |2       |2026-10-01|Té verde                        |Tés e infusiones |4       |3.50  |14.00        |Efectivo   |
|4            |3       |2026-10-01|Café americano                  |Cafés            |6       |4.00  |24.00        |Efectivo   |
|5            |4       |2026-10-01|Té verde                        |Tés e infusiones |2       |3.

**Paso 10. Agrupar y analizar las ventas**

In [0]:
df_ventas_categoria = df.groupBy("categoria").sum("cantidad", "importe_total")
display(df_ventas_categoria)

categoria,sum(cantidad),sum(importe_total)
Pasteles y tortas,254,1643.50
Tés e infusiones,2721,8668.50
Cafés,3137,12984.50
Postres y dulces,289,1045.50
Desayunos,142,1237.00
Snacks y aperitivos,128,413.00
Galletas y bocaditos,386,1319.50
Bebidas frías,225,839.50
Panes,254,647.30
Jugos y bebidas naturales,222,1522.50


In [0]:
df_ventas_pago = df.groupBy("metodo_pago").sum("importe_total")
display(df_ventas_pago)

metodo_pago,sum(importe_total)
Efectivo,23353.90
Yape,7199.70
Plin,2183.90
Tarjeta,528.30


In [0]:
df_cafes = df.filter(
    col("categoria") == "Cafés"
).select(
    "producto",
    "cantidad",
    "precio",
    "importe_total"
)
display(df_cafes)

producto,cantidad,precio,importe_total
Café americano,6,4.00,24.00
Café americano,3,4.00,12.00
Café cappuccino,1,7.00,7.00
Café americano,10,4.00,40.00
Café americano,6,4.00,24.00
Café latte,1,7.50,7.50
Café americano,4,4.00,16.00
Café americano,4,4.00,16.00
Café espresso,2,3.50,7.00
Café latte,1,7.50,7.50


In [0]:
df.groupBy("producto") \
    .sum("cantidad", "importe_total") \
    .orderBy(col("sum(importe_total)").desc()) \
    .show(10, truncate=False)

+-------------------------+-------------+------------------+
|producto                 |sum(cantidad)|sum(importe_total)|
+-------------------------+-------------+------------------+
|Café americano           |2955         |11820.00          |
|Infusión de anís         |1630         |4890.00           |
|Té verde                 |881          |3083.50           |
|Sándwich de chicharrón   |55           |550.00            |
|Jugo surtido             |66           |528.00            |
|Sándwich de pollo y palta|58           |522.00            |
|Sándwich de pollo        |65           |520.00            |
|Cheesecake de fresa      |54           |432.00            |
|Hamburguesa clásica      |43           |430.00            |
|Empanada de pollo        |94           |423.00            |
+-------------------------+-------------+------------------+
only showing top 10 rows


**CREACIÓN DE BASE DE DATOS Y TABLA PERSISTENTE**

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS villarica;

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("villarica.tb_datosAnalisis_villarica")

dfVentasGuardadas = spark.table("villarica.tb_datosAnalisis_villarica")
display(dfVentasGuardadas)

id_fact_venta,id_venta,fecha,producto,categoria,cantidad,precio,importe_total,metodo_pago
1,1,2026-10-01,Torta de chocolate,Pasteles y tortas,1,7.00,7.00,Efectivo
2,1,2026-10-01,Keke de naranja,Pasteles y tortas,1,4.00,4.00,Efectivo
3,2,2026-10-01,Té verde,Tés e infusiones,4,3.50,14.00,Efectivo
4,3,2026-10-01,Café americano,Cafés,6,4.00,24.00,Efectivo
5,4,2026-10-01,Té verde,Tés e infusiones,2,3.50,7.00,Yape
6,4,2026-10-01,Galleta con chispas de chocolate,Postres y dulces,1,3.00,3.00,Yape
7,5,2026-10-01,Café americano,Cafés,3,4.00,12.00,Efectivo
8,6,2026-10-01,Café cappuccino,Cafés,1,7.00,7.00,Yape
9,7,2026-10-01,Té verde,Tés e infusiones,4,3.50,14.00,Efectivo
10,7,2026-10-01,Pan con huevo,Desayunos,1,5.00,5.00,Efectivo


**ACÁ YA HARÉ MIS VISUALIZACIONES**

In [0]:
dfVentasCategoria = df.groupBy("categoria").sum("importe_total")
display(dfVentasCategoria)

categoria,sum(importe_total)
Pasteles y tortas,1643.50
Tés e infusiones,8668.50
Cafés,12984.50
Postres y dulces,1045.50
Desayunos,1237.00
Snacks y aperitivos,413.00
Galletas y bocaditos,1319.50
Bebidas frías,839.50
Panes,647.30
Jugos y bebidas naturales,1522.50


Databricks visualization. Run in Databricks to view.

In [0]:
dfProductosVendidos = (df.groupBy("producto")
    .sum("cantidad")
    .orderBy(col("sum(cantidad)").desc())
    .limit(10)
)
display(dfProductosVendidos)

producto,sum(cantidad)
Café americano,2955
Infusión de anís,1630
Té verde,881
Galletas de vainilla,119
Empanada de pollo,94
Enrollado de hot dog,78
Té de manzanilla,74
Té de hierba luisa,71
Jugo surtido,66
Sándwich de pollo,65


Databricks visualization. Run in Databricks to view.

In [0]:
dfVentasPago = df.groupBy("metodo_pago").sum("importe_total")
display(dfVentasPago)

metodo_pago,sum(importe_total)
Efectivo,23353.90
Yape,7199.70
Plin,2183.90
Tarjeta,528.30


Databricks visualization. Run in Databricks to view.

**EXPERIMENTO 1: Big Data – Evaluación del rendimiento mediante particiones en Apache Spark**

In [0]:
import time
from pyspark.sql.functions import col, sum as spark_sum

# Escenarios de particionamiento
particiones = [1, 2, 4]
resultados_particiones = []

for numero in particiones:
    inicio = time.perf_counter()

    # Distribuir los datos en diferentes particiones
    df_particionado = df.repartition(numero)

    # Ejecutar agrupación y cálculo de ventas
    resultado = df_particionado.groupBy("categoria").agg(
        spark_sum("cantidad").alias("Total_Unidades"),
        spark_sum(col("cantidad") * col("precio")).alias("Total_Ventas")
    )

    # Ejecutar el procesamiento
    resultado.collect()

    fin = time.perf_counter()
    resultados_particiones.append((
        numero,
        round(fin - inicio, 3)
    ))

# Crear tabla comparativa
df_particiones = spark.createDataFrame(
    resultados_particiones,
    ["Numero_Particiones", "Tiempo_Segundos"]
)

display(df_particiones)

Numero_Particiones,Tiempo_Segundos
1,0.683
2,0.778
4,0.813


Databricks visualization. Run in Databricks to view.

In [0]:
import time
from functools import reduce

# Dataset base
df_base = df

# Volúmenes de prueba
multiplicadores = [1, 2, 5, 10]
resultados = []

for factor in multiplicadores:
    # Generar volumen de prueba mediante UNION
    data_prueba = reduce(
        lambda a, b: a.union(b),
        [df_base] * factor
    )

    inicio = time.perf_counter()

    # Ejecutar procesamiento
    total_registros = data_prueba.count()

    fin = time.perf_counter()
    tiempo = fin - inicio

    resultados.append((
        factor,
        total_registros,
        round(tiempo, 3)
    ))

# Mostrar resultados
columnas = [
    "Multiplicador",
    "Total_Registros",
    "Tiempo_Segundos"
]

df_resultados = spark.createDataFrame(
    resultados, columnas
)

display(df_resultados)

Multiplicador,Total_Registros,Tiempo_Segundos
1,3500,0.442
2,7000,0.639
5,17500,0.927
10,35000,1.15


Databricks visualization. Run in Databricks to view.

**EXPERIMENTO 2: BI – Análisis de la evolución de ventas por día**

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Asegurar importe_total
df_bi = df.withColumn(
    "importe_total",
    col("cantidad") * col("precio")
)

# 1. Ventas por categoría
ventas_categoria = df_bi.groupBy("categoria").agg(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Ventas").desc())

display(ventas_categoria)

# 2. Ventas por método de pago
ventas_pago = df_bi.groupBy("metodo_pago").agg(
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Ventas").desc())

display(ventas_pago)

# 3. Productos más vendidos
productos_vendidos = df_bi.groupBy("producto").agg(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Unidades").desc()).limit(10)

display(productos_vendidos)

categoria,Total_Unidades,Total_Ventas
Cafés,3137,12984.50
Tés e infusiones,2721,8668.50
Sándwiches y hamburguesas,263,2316.00
Pasteles y tortas,254,1643.50
Jugos y bebidas naturales,222,1522.50
Galletas y bocaditos,386,1319.50
Desayunos,142,1237.00
Postres y dulces,289,1045.50
Bebidas frías,225,839.50
Panes,254,647.30


Databricks visualization. Run in Databricks to view.

metodo_pago,Total_Ventas
Efectivo,23353.90
Yape,7199.70
Plin,2183.90
Tarjeta,528.30


Databricks visualization. Run in Databricks to view.

producto,Total_Unidades,Total_Ventas
Café americano,2955,11820.00
Infusión de anís,1630,4890.00
Té verde,881,3083.50
Galletas de vainilla,119,178.50
Empanada de pollo,94,423.00
Enrollado de hot dog,78,312.00
Té de manzanilla,74,222.00
Té de hierba luisa,71,213.00
Jugo surtido,66,528.00
Sándwich de pollo,65,520.00


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Calcular importe total
df_evolucion = df.withColumn(
    "importe_total",
    col("cantidad") * col("precio")
)

# Agrupar las ventas por fecha
ventas_diarias = df_evolucion.groupBy("fecha").agg(
    spark_sum("importe_total").alias("Total_Ventas"),
    spark_sum("cantidad").alias("Total_Unidades")
).orderBy("fecha")

# Mostrar resultados
display(ventas_diarias)

fecha,Total_Ventas,Total_Unidades
2026-10-01,3572.40,886
2026-10-02,4059.60,986
2026-10-03,3854.60,934
2026-10-04,5034.40,1226
2026-10-05,3964.20,966
2026-10-06,4327.20,1040
2026-10-07,2125.40,523
2026-10-08,2023.20,506
2026-10-09,2164.20,536
2026-10-10,2140.60,517


Databricks visualization. Run in Databricks to view.